In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip -q install "transformers>=4.44,<5" "datasets>=2.20,<4" "trl>=0.9,<1" "peft>=0.12,<1" "bitsandbytes>=0.43,<1" "accelerate>=0.34,<2" "sentencepiece>=0.2,<1" "pandas==2.2.2" "numpy==1.26.4" "scipy>=1.11,<1.14"


### SummEval 1.5B Batch 3


In [ ]:
from pathlib import Path

BASE_DIR = Path('/content/drive/MyDrive/llm_judge/model1.5b')
for path in [
    BASE_DIR,
    BASE_DIR / 'summeval_batch3_600.jsonl',
    BASE_DIR / 'prompt.txt',
    BASE_DIR / 'summeval_gold_flat.jsonl',
    Path('/content/drive/MyDrive/llm_judge/artifacts/qwen25_1.5b_lora_judge/final'),
]:
    print(path, path.exists())


In [ ]:
#!/usr/bin/env python
"""Run batched inference with Qwen2.5-1.5B LoRA judge and save predictions to JSONL."""

import json
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

LIMIT = 0
BATCH_SIZE = 8
max_new_tokens = 320
DEFAULT_BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
BASE_DIR = Path("/content/drive/MyDrive/llm_judge/model1.5b")
DEFAULT_ADAPTER_PATH = Path("/content/drive/MyDrive/llm_judge/artifacts/qwen25_1.5b_lora_judge/final")
DEFAULT_INPUT_PATH = BASE_DIR / "summeval_batch3_600.jsonl"
DEFAULT_OUTPUT_PATH = BASE_DIR / "qwen25_1.5b_summeval_batch3_preds.jsonl"
PROMPT_PATH = BASE_DIR / "prompt.txt"
REQUIRED_KEYS = ("coherence", "consistency", "fluency", "relevance")


def build_input(base_prompt: str, article: str, summary: str) -> str:
    return (
        f"{base_prompt}\n\n"
        "This is the complete source article for which you will be evaluating:\n"
        f"{article}\n\n"
        "This is the LLM generated summary you are about to evaluate:\n"
        f"{summary}\n\n"
        "Return ONLY the final JSON object."
    )


def load_rows(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as fin:
        for line in fin:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def _clean_explanation_text(text: str) -> str:
    end_markers = (
        "\n\nThis is the complete source article",
        "\n\nThis is the LLM generated summary you are about to evaluate",
        "\n\nReturn ONLY the final JSON object.",
    )
    for marker in end_markers:
        if marker in text:
            text = text.split(marker, 1)[0]
    text = text.strip()
    suffixes = ('"]}', '"]}', '"]', '"}', '"}', '"},', '},', '},', '}', ',')
    changed = True
    while changed:
        changed = False
        stripped = text.rstrip()
        for suffix in suffixes:
            if stripped.endswith(suffix):
                stripped = stripped[: -len(suffix)].rstrip()
                changed = True
        text = stripped
    return text.replace('" "', ' ').replace('\\\"', '"').strip()


def _extract_dimension_from_text(raw: str, dim: str, next_dim: str | None) -> dict | None:
    start_marker = f'"{dim}": {{'
    start = raw.find(start_marker)
    if start == -1:
        return None
    if next_dim is None:
        block = raw[start:]
    else:
        next_marker = f'"{next_dim}": {{'
        end = raw.find(next_marker, start + len(start_marker))
        if end == -1:
            return None
        block = raw[start:end]

    score_marker = '"score":'
    score_pos = block.find(score_marker)
    if score_pos == -1:
        return None
    i = score_pos + len(score_marker)
    while i < len(block) and block[i] in " \t":
        i += 1
    score_chars = []
    while i < len(block) and block[i].isdigit():
        score_chars.append(block[i])
        i += 1
    if not score_chars:
        return None

    explanation_marker = '"explanation": "'
    explanation_pos = block.find(explanation_marker)
    if explanation_pos == -1:
        return None
    explanation = block[explanation_pos + len(explanation_marker):]
    explanation = _clean_explanation_text(explanation)
    if not explanation:
        return None

    return {
        "score": int("".join(score_chars)),
        "explanation": explanation,
    }


def _schema_extract_json(raw: str) -> dict | None:
    parsed = {}
    for idx, dim in enumerate(REQUIRED_KEYS):
        next_dim = REQUIRED_KEYS[idx + 1] if idx + 1 < len(REQUIRED_KEYS) else None
        item = _extract_dimension_from_text(raw, dim, next_dim)
        if item is None:
            return None
        parsed[dim] = item
    return parsed


def extract_json(raw: str) -> dict | None:
    raw = (raw or "").strip()
    if not raw:
        return None

    candidates = []
    for candidate in (raw, raw[raw.find("{"):] if "{" in raw else raw):
        if candidate and candidate not in candidates:
            candidates.append(candidate)
        if candidate:
            last_brace = candidate.rfind("}")
            if last_brace != -1:
                cropped = candidate[: last_brace + 1]
                if cropped and cropped not in candidates:
                    candidates.append(cropped)

    for candidate in candidates:
        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            parsed = _schema_extract_json(candidate)
            if parsed is not None:
                return parsed
    return None


def has_complete_scores(prediction_json: dict | None) -> bool:
    if not isinstance(prediction_json, dict):
        return False
    for dim in REQUIRED_KEYS:
        item = prediction_json.get(dim)
        if not isinstance(item, dict):
            return False
        if item.get("score") is None:
            return False
    return True


def get_prompt_for_row(row: dict, base_prompt: str) -> str:
    if isinstance(row.get("messages"), list) and row["messages"]:
        for message in row["messages"]:
            if message.get("role") == "user":
                return message.get("content", "")

    article = row.get("original", "")
    summary = row.get("summary", "")
    if article and summary:
        return build_input(base_prompt, article, summary)

    raise ValueError("Row missing both messages and original+summary.")


base_prompt = PROMPT_PATH.read_text(encoding="utf-8")

input_path = Path(DEFAULT_INPUT_PATH)
output_path = Path(DEFAULT_OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)

rows = load_rows(input_path)
if LIMIT > 0:
    rows = rows[:LIMIT]

dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
tokenizer = AutoTokenizer.from_pretrained(DEFAULT_BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    DEFAULT_BASE_MODEL,
    dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)
model = PeftModel.from_pretrained(base_model, DEFAULT_ADAPTER_PATH)
model.eval()

total = len(rows)
with output_path.open("w", encoding="utf-8") as fout:
    for start_idx in range(0, total, BATCH_SIZE):
        batch_rows = rows[start_idx : start_idx + BATCH_SIZE]
        rendered_messages = []
        for row in batch_rows:
            prompt = get_prompt_for_row(row, base_prompt)
            rendered_messages.append(
                tokenizer.apply_chat_template(
                    [{"role": "user", "content": prompt}],
                    add_generation_prompt=True,
                    tokenize=False,
                )
            )

        inputs = tokenizer(
            rendered_messages,
            padding=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        generated_tokens = outputs[:, inputs["input_ids"].shape[1]:]
        pred_texts = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

        for row, pred_text in zip(batch_rows, pred_texts):
            pred_text = pred_text.strip()
            pred_json = extract_json(pred_text)

            out = {
                "id": row["id"],
                "article_id": row.get("article_id"),
                "summary_index": row.get("summary_index"),
                "original": row.get("original"),
                "summary": row.get("summary"),
                "gold_relevance": row.get("gold_relevance"),
                "gold_coherence": row.get("gold_coherence"),
                "gold_fluency": row.get("gold_fluency"),
                "gold_consistency": row.get("gold_consistency"),
                "prediction_text": pred_text,
                "prediction_json": pred_json,
                "parse_ok": has_complete_scores(pred_json),
            }
            fout.write(json.dumps(out, ensure_ascii=False) + "\n")

        processed = start_idx + len(batch_rows)
        print(f"[{processed}/{total}] done")

print("saved to", DEFAULT_OUTPUT_PATH)


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import kendalltau, pearsonr, spearmanr

BASE_DIR = Path("/content/drive/MyDrive/llm_judge/model1.5b")
PRED_PATH = BASE_DIR / "qwen25_1.5b_summeval_batch3_preds.jsonl"
GOLD_PATH = BASE_DIR / "summeval_gold_flat.jsonl"
OUT_PATH = BASE_DIR / "eval_tables_summeval_qwen15_batch3.csv"
DIMENSIONS = ["coherence", "consistency", "fluency", "relevance"]


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def _clean_explanation_text(text):
    end_markers = (
        "\n\nThis is the complete source article",
        "\n\nThis is the LLM generated summary you are about to evaluate",
        "\n\nReturn ONLY the final JSON object.",
    )
    for marker in end_markers:
        if marker in text:
            text = text.split(marker, 1)[0]
    text = text.strip()
    suffixes = ('"]}', '"]}', '"]', '"}', '"}', '"},', '},', '},', '}', ',')
    changed = True
    while changed:
        changed = False
        stripped = text.rstrip()
        for suffix in suffixes:
            if stripped.endswith(suffix):
                stripped = stripped[: -len(suffix)].rstrip()
                changed = True
        text = stripped
    return text.replace('" "', ' ').replace('\\\"', '"').strip()


def _extract_dimension_from_text(raw, dim, next_dim):
    start_marker = f'"{dim}": {{'
    start = raw.find(start_marker)
    if start == -1:
        return None
    if next_dim is None:
        block = raw[start:]
    else:
        next_marker = f'"{next_dim}": {{'
        end = raw.find(next_marker, start + len(start_marker))
        if end == -1:
            return None
        block = raw[start:end]

    score_marker = '"score":'
    score_pos = block.find(score_marker)
    if score_pos == -1:
        return None
    i = score_pos + len(score_marker)
    while i < len(block) and block[i] in " \t":
        i += 1
    score_chars = []
    while i < len(block) and block[i].isdigit():
        score_chars.append(block[i])
        i += 1
    if not score_chars:
        return None

    explanation_marker = '"explanation": "'
    explanation_pos = block.find(explanation_marker)
    if explanation_pos == -1:
        return None
    explanation = block[explanation_pos + len(explanation_marker):]
    explanation = _clean_explanation_text(explanation)
    if not explanation:
        return None

    return {
        "score": int("".join(score_chars)),
        "explanation": explanation,
    }


def _schema_extract_json(raw):
    parsed = {}
    for idx, dim in enumerate(DIMENSIONS):
        next_dim = DIMENSIONS[idx + 1] if idx + 1 < len(DIMENSIONS) else None
        item = _extract_dimension_from_text(raw, dim, next_dim)
        if item is None:
            return None
        parsed[dim] = item
    return parsed


def try_parse_json(x):
    if x is None:
        return None
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None
        try:
            return json.loads(x)
        except:
            start = x.find("{")
            end = x.rfind("}")
            if start != -1 and end != -1 and end > start:
                repaired = _schema_extract_json(x[start:end+1])
                if repaired is not None:
                    return repaired
            repaired = _schema_extract_json(x)
            if repaired is not None:
                return repaired
    return None


def extract_scores(obj):
    obj = try_parse_json(obj)
    out = {}
    for dim in DIMENSIONS:
        score = None
        if isinstance(obj, dict) and isinstance(obj.get(dim), dict):
            score = obj[dim].get("score")
        out[dim] = score
    return out


pred_rows = load_jsonl(PRED_PATH)

pred_data = []
gold_data = []
for row in pred_rows:
    pred_source = row.get("prediction_json")
    if pred_source is None:
        pred_source = row.get("prediction_text")
    pred_scores = extract_scores(pred_source)

    pred_rec = {"id": str(row.get("id"))}
    gold_rec = {"id": str(row.get("id"))}

    for dim in DIMENSIONS:
        pred_rec[f"pred_{dim}"] = pred_scores[dim]
        gold_rec[f"gold_{dim}"] = row.get(f"gold_{dim}")

    pred_data.append(pred_rec)
    gold_data.append(gold_rec)

pred_df = pd.DataFrame(pred_data)
gold_df = pd.DataFrame(gold_data)
merged = pred_df.merge(gold_df, on="id", how="inner")

for dim in DIMENSIONS:
    merged[f"pred_{dim}"] = pd.to_numeric(merged[f"pred_{dim}"], errors="coerce")
    merged[f"gold_{dim}"] = pd.to_numeric(merged[f"gold_{dim}"], errors="coerce")

usable_counts = {}
for dim in DIMENSIONS:
    mask = ~(pd.isna(merged[f"pred_{dim}"]) | pd.isna(merged[f"gold_{dim}"]))
    usable_counts[dim] = int(mask.sum())


def safe_corr(fn, x, y):
    mask = ~(pd.isna(x) | pd.isna(y))
    x = np.array(x[mask], dtype=float)
    y = np.array(y[mask], dtype=float)
    if len(x) < 2:
        return np.nan
    try:
        return fn(x, y)[0]
    except:
        return np.nan


def evaluate_dimension(df, dim):
    pred = df[f"pred_{dim}"]
    gold = df[f"gold_{dim}"]

    mask = ~(pd.isna(pred) | pd.isna(gold))
    pred = np.array(pred[mask], dtype=float)
    gold = np.array(gold[mask], dtype=float)

    if len(pred) == 0:
        return {
            "dimension": dim,
            "n": 0,
            "spearman": np.nan,
            "pearson": np.nan,
            "kendall": np.nan,
            "mae": np.nan,
            "rmse": np.nan,
            "exact_match": np.nan,
            "off_by_1_acc": np.nan,
            "pred_mean": np.nan,
            "gold_mean": np.nan,
        }

    return {
        "dimension": dim,
        "n": len(pred),
        "spearman": safe_corr(spearmanr, pred, gold),
        "pearson": safe_corr(pearsonr, pred, gold),
        "kendall": safe_corr(kendalltau, pred, gold),
        "mae": np.mean(np.abs(pred - gold)),
        "rmse": np.sqrt(np.mean((pred - gold) ** 2)),
        "exact_match": np.mean(pred == gold),
        "off_by_1_acc": np.mean(np.abs(pred - gold) <= 1),
        "pred_mean": np.mean(pred),
        "gold_mean": np.mean(gold),
    }


eval_df = pd.DataFrame([evaluate_dimension(merged, dim) for dim in DIMENSIONS])
eval_df_rounded = eval_df.copy()
for col in ["spearman", "pearson", "kendall", "mae", "rmse", "exact_match", "off_by_1_acc", "pred_mean", "gold_mean"]:
    eval_df_rounded[col] = eval_df_rounded[col].round(4)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
eval_df_rounded.to_csv(OUT_PATH, index=False)

print("pred rows:", len(pred_df))
print("gold rows:", len(gold_df))
print("merged rows:", len(merged))
print("usable rows by dimension:", usable_counts)
print(eval_df_rounded)
print("saved to:", OUT_PATH)
